# D01–D02 — Chốt Knowledge Base thật của project

Notebook này **không cần GPU** và **không gọi model**. Nguồn D01–D02 là chính `data/kb/raw_docs/gout_guideline_1.pdf` đã được đóng gói trong project.

D02 cho phép dùng nguồn này cho RAG của project nhưng **không tự tuyên bố mọi khuyến cáo năm 2014 là mới nhất năm 2026**.


In [ ]:
from google.colab import files
from pathlib import Path
import io, zipfile, tempfile, subprocess, sys, json
uploaded = files.upload()
archives = [(n,b) for n,b in uploaded.items() if n.lower().endswith('.zip')]
if len(archives) != 1:
    raise ValueError('Chọn đúng một file Gout_Build_D01_D02_KB_v08.zip')
WORK = Path(tempfile.mkdtemp(prefix='gout_d01d02_'))
with zipfile.ZipFile(io.BytesIO(archives[0][1])) as z:
    for item in z.infolist():
        target = (WORK / item.filename).resolve()
        if not target.is_relative_to(WORK.resolve()):
            raise ValueError('ZIP path không hợp lệ')
    z.extractall(WORK)
PROJECT = WORK / 'gout_units_v01'
print(PROJECT)


In [ ]:
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', str(PROJECT/'tests'), '-p', 'test_*.py', '-v'], cwd=PROJECT, check=True)


In [ ]:
OUT = WORK / 'd01_d02_run'
subprocess.run([sys.executable, str(PROJECT/'unit04_sources.py'), '--out', str(OUT)], cwd=PROJECT, check=True)
print((OUT/'manifest.json').read_text(encoding='utf-8'))
print((OUT/'source_registry.jsonl').read_text(encoding='utf-8'))
print((OUT/'review_decision.json').read_text(encoding='utf-8'))


Kết quả mong đợi: `d01_completed=true`, `d02_completed_source_scope=true`, `source_allowed_for_rag=true`, `currentness_claimed=false`. RAG chưa chạy ở đây; D03 mới trích text.
